In [1]:
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

# Install dependencies
!pip install langchain langchain-mistralai langsmith -q

# LangSmith observability setup
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_PROJECT"] = "FinSight_LoanDecider"

print("LangSmith tracing enabled")
print(f"Project: FinSight_LoanDecider")

Mounted at /content/drive
LangSmith tracing enabled
Project: FinSight_LoanDecider


In [4]:
# Cell 2 — load scored data and initialize Mistral:
import pandas as pd
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate

import joblib

# Load scored test set
df_scored = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D2_Classifier/finsight_v2_scored.csv')

# Load full dataset for original columns
df = pd.read_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/Data/loan_data_clean.csv')

print(f"Test set: {df_scored.shape}")
print(f"Full dataset: {df.shape}")
print(f"\nRisk tier distribution:")
print(df_scored['risk_tier'].value_counts())

# Filter high risk only
high_risk_df = df_scored[df_scored['risk_tier'] == 'High'].reset_index(drop=True)
print(f"Total applications: {len(df_scored)}")
print(f"High risk applications: {len(high_risk_df)}")
print(f"\nSample high risk profile:")
print(high_risk_df[['loan_id','loan_grade','person_income',
                     'loan_amnt','loan_intent','default_probability']].head(3))

# Initialize Mistral
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=userdata.get('MISTRAL_API_KEY'),
    temperature=0.35
)

# Test connection
test = llm.invoke("Reply with: FinSight LoanDecider connected.")
print(f"\n{test.content}")

Test set: (6515, 15)
Full dataset: (32574, 14)

Risk tier distribution:
risk_tier
Low       5069
High      1082
Medium     364
Name: count, dtype: int64
Total applications: 6515
High risk applications: 1082

Sample high risk profile:
   loan_id  loan_grade  person_income  loan_amnt  loan_intent  \
0  LN03671           3          13000       4000            4   
1  LN28641           0          45000      14000            4   
2  LN21466           2          42000       5000            1   

   default_probability  
0             0.939322  
1             0.898735  
2             0.605773  

FinSight LoanDecider connected.


In [7]:
high_risk_pool = df_scored[df_scored['risk_tier'] == 'High'].copy()

tier_60_75 = high_risk_pool[high_risk_pool['default_probability'].between(0.60, 0.75)].sample(min(50, len(high_risk_pool[high_risk_pool['default_probability'].between(0.60, 0.75)])), random_state=42)
tier_75_90 = high_risk_pool[high_risk_pool['default_probability'].between(0.75, 0.90)].sample(min(50, len(high_risk_pool[high_risk_pool['default_probability'].between(0.75, 0.90)])), random_state=42)
tier_90_95 = high_risk_pool[high_risk_pool['default_probability'].between(0.90, 0.95)].sample(min(50, len(high_risk_pool[high_risk_pool['default_probability'].between(0.90, 0.95)])), random_state=42)
tier_95_100 = high_risk_pool[high_risk_pool['default_probability'] >= 0.95].sample(min(50, len(high_risk_pool[high_risk_pool['default_probability'] >= 0.95])), random_state=42)

high_risk_sample = pd.concat([tier_60_75, tier_75_90, tier_90_95, tier_95_100]).reset_index(drop=True)

print(f"Stratified sample: {len(high_risk_sample)} applications")
print(f"\nProbability distribution of sample:")
print(high_risk_sample['default_probability'].describe().round(3))

Stratified sample: 150 applications

Probability distribution of sample:
count    150.000
mean       0.816
std        0.112
min        0.600
25%        0.720
50%        0.854
75%        0.921
max        0.941
Name: default_probability, dtype: float64


In [9]:
original_cols = [ 'loan_id', 'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
                  'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
                  'cb_person_default_on_file', 'person_home_ownership',
                  'loan_intent', 'loan_grade']

high_risk_alerts = df_scored[df_scored['loan_id'].isin(high_risk_sample['loan_id'])][original_cols].copy()
high_risk_alerts = high_risk_alerts.merge(
    high_risk_sample[['loan_id', 'default_probability', 'risk_tier']],
    on='loan_id', how='left').reset_index(drop=True)

print(f"Alert candidates: {len(high_risk_alerts)}")
print(high_risk_alerts[['loan_id','loan_grade','person_income',
                         'loan_amnt','default_probability']].head(3))

Alert candidates: 150
   loan_id  loan_grade  person_income  loan_amnt  default_probability
0  LN09989           0          29000       9200             0.936055
1  LN16880           3          18000       5400             0.831999
2  LN25483           3          36000       3000             0.829756


In [10]:

# Prompt template
alert_template = PromptTemplate(
    input_variables=["loan_id", "person_age", "person_income",
                     "person_emp_length", "loan_amnt", "loan_int_rate",
                     "loan_percent_income", "cb_person_cred_hist_length", "cb_person_default_on_file",
                     "person_home_ownership","loan_intent","loan_grade"],
    template="""
You are a senior credit analyst at FinSight Capital writing an internal note.

Write a 3-sentence alert for the branch manager:
- Sentence 1: State the risk tier and loan purpose, explain why this application is concerning
- Sentence 2: Identify the two most concerning factors with specific numbers
- Sentence 3: Recommend one specific action before approval

Applicant Profile:
- Loan ID: {loan_id}
- Age: {person_age} years old
- Annual Income: ${person_income} | Employment Experience: {person_emp_length} years
- Home Ownership: {person_home_ownership}
- Credit History Length: {cb_person_cred_hist_length} years
- Default Status: {cb_person_default_on_file}
- Loan Amount: ${loan_amnt} | Purpose: {loan_intent} | Interest Rate: {loan_int_rate}% | Percent Income: {loan_percent_income}%
- Loan Grade: {loan_grade}

Rules:
- Do NOT use the words: model, algorithm, AI
- Write in flowing prose, no bullet points, no bold headers
- Write exactly 3 sentences
- 100 words maximum
- Read like a human credit analyst note
- Write in plain text only — no asterisks, no bold, no italic, no headers
- Do not start with "Alert" or "Credit Alert" or any label
- First word must be a regular English word starting the first sentence directly
"""
)

alert_chain = alert_template | llm
print("Prompt template ready")

Prompt template ready


In [11]:
print(df_scored['default_probability'].value_counts().head(10))
print(f"\nUnique probability values: {df_scored['default_probability'].nunique()}")
print(f"\nProbability distribution:")
print(df_scored['default_probability'].describe())

default_probability
0.018166    56
0.018436    41
0.018448    36
0.018436    29
0.018718    28
0.018718    25
0.941150    24
0.019270    17
0.018988    16
0.018706    15
Name: count, dtype: int64

Unique probability values: 5320

Probability distribution:
count    6515.000000
mean        0.217372
std         0.309764
min         0.018166
25%         0.026821
50%         0.053209
75%         0.231425
max         0.941150
Name: default_probability, dtype: float64


In [14]:
# INvoking and creating alert test from mistral AI for high risk tier of 200 users

import time

alerts = []

for idx, row in high_risk_alerts.iterrows():
    while True:
        try:
            response = alert_chain.invoke({
              "loan_id": row['loan_id'],
              "person_age": row['person_age'],
              "person_income": f"{int(row['person_income']):,}",
              "person_emp_length": row['person_emp_length'],
              "person_home_ownership": row['person_home_ownership'],
              "cb_person_cred_hist_length": row['cb_person_cred_hist_length'],
              "cb_person_default_on_file": row['cb_person_default_on_file'],
              "loan_amnt": f"{int(row['loan_amnt']):,}",
              "loan_intent": row['loan_intent'],
              "loan_int_rate": row['loan_int_rate'],
              "loan_percent_income": row['loan_percent_income'],
              "loan_grade": row['loan_grade']
            })
            alerts.append({
                "loan_id": row['loan_id'],
                "default_probability": row['default_probability'],
                "risk_tier": row['risk_tier'],
                "alert_text": response.content
            })
            time.sleep(1)
            break
        except Exception as e:
            if '429' in str(e):
                print(f"Rate limit at {row['loan_id']}. Waiting 60s...")
                time.sleep(60)
            else:
                print(f"Error on {row['loan_id']}: {e}")
                break

    # Checkpoint save every 20 alerts
    if len(alerts) % 20 == 0 and len(alerts) > 0:
        pd.DataFrame(alerts).to_csv(
            '/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D3_Alerts/alerts_checkpoint.csv',
            index=False)
        print(f"Checkpoint saved: {len(alerts)} alerts")

print(f"\nDone. Total alerts: {len(alerts)}")

Checkpoint saved: 20 alerts
Checkpoint saved: 40 alerts
Checkpoint saved: 60 alerts
Checkpoint saved: 80 alerts
Checkpoint saved: 100 alerts
Checkpoint saved: 120 alerts
Checkpoint saved: 140 alerts

Done. Total alerts: 150


In [15]:
# Preview 3 alerts across probability range
for i in [0, 100, 149]:
    print(f"\n--- Alert {i+1} | {alerts[i]['loan_id']} | Prob: {alerts[i]['default_probability']:.3f} ---")
    print(alerts[i]['alert_text'])
    print()


--- Alert 1 | LN09989 | Prob: 0.936 ---
This application falls into the high-risk tier due to the borrower’s short credit history and the purpose of the loan being unspecified. The most concerning factors are the borrower’s limited credit history of just three years and the loan amount representing 32% of their annual income, which is well above prudent lending thresholds. Before approval, I recommend requesting additional documentation to clarify the loan purpose and verify income stability.


--- Alert 101 | LN04727 | Prob: 0.608 ---
This application falls into the high-risk tier due to the borrower’s limited credit history and modest income relative to the loan size. The two most concerning factors are the 23-year-old applicant’s just two years of credit history and the $5,000 loan representing 17% of their $29,100 annual income, which strains affordability despite the low default status. Before approval, require a co-signer with stronger credit to mitigate repayment risk.


--- Al

In [16]:
alerts_df_final = pd.DataFrame(alerts)

# Merge with applicant details
final_output = high_risk_alerts.merge(
    alerts_df_final[['loan_id', 'alert_text']],
    on='loan_id', how='left')

# Save
alerts_df_final.to_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D3_Alerts/finsight_alerts_final.csv', index=False)
final_output.to_csv('/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D3_Alerts/finsight_high_risk_with_alerts.csv', index=False)

print(f"Alerts saved: {len(alerts_df_final)}")
print(f"Full output saved: {len(final_output)}")

Alerts saved: 150
Full output saved: 150
